In [ ]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import pandas as pd

from time import sleep

import datetime

from bs4 import BeautifulSoup

from pandas import ExcelWriter

from selenium import webdriver

from selenium.webdriver.common.by import By

import os

from selenium.webdriver.common.keys import Keys

from zipfile import ZipFile



# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'ZA FSCA' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.1.1")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

#scriptfolder = f"C:\\Users\\siewekoa\\OneDrive - moodys.com\\Desktop\\My_data\\Project_work\\scripts_regulator\\{regulatorName}" ## to comment for the production environment

scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)



# %%

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert

         }

chromeOptions.add_argument("--disable-search-engine-choice-screen")

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict = {

            'ZA FSCA 1': 'https://www.fsca.co.za/MagicScripts/mgrqispi.dll?APPNAME=Web&PRGNAME=List_Of_Registered_Insurers',

            'ZA FSCA 2': 'https://www.fsca.co.za/MagicScripts/mgrqispi.dll?APPNAME=Web&PRGNAME=Search_Mancos',

            'ZA FSCA 3': 'https://www.fsca.co.za/MagicScripts/mgrqispi.dll?APPNAME=Web&PRGNAME=Search_Mancos',

            'ZA FSCA 4': 'https://www.fsca.co.za/Regulated%20Entities/Pages/Credit-Rating-Agencies.aspx',

            'ZA FSCA 5': 'https://www.fsca.co.za/Regulated%20Entities/Pages/LREP-Retirement-Fund-Registered-Active-Funds.aspx',

           }

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

		  'Phone - Mother company': [], 'Check': []}

columns = ['Payment institution', 'Competent', 'Method', 'Payment', 'Date']

processdate = now.strftime('%Y-%m-%d')



# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    len_value=[]

    for key, value in sqldict.items():

        len_value.append(len(value))

    maxlen = max(len_value)

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



def check_dowload_files(tempfolder, fileType, wait_time=10):

    for time in range(wait_time):

        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele]) != 0 :

            print(f"[INFO] : - {fileType} file = {os.listdir(tempfolder)})")

            break

        else:

            print(f"[INFO] : - Download {fileType} file ... (wait {time*2}/20 s)")

            sleep(2)

    else:

        raise Exception(f'[ERROR] : - Failed to Download {fileType} file. Run Script again' )

    return  os.listdir(tempfolder)[0]



def unzip(source_path, output_path):

    with ZipFile(source_path, 'r') as zip_:

        zip_.extractall(output_path) 



# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):

    print(f"[INFO] : Working {k+1}/{len(regdict)} | {reg}")

    driver.get(regdict[reg])

    sleep(1)

    soup=BeautifulSoup(driver.page_source, 'html.parser')

    sleep(0.2)



    if reg=='ZA FSCA 1'  :

        for br in soup.find_all("br"):

            br.replace_with("|")

        tbody = soup.find_all('tbody')[-1].find_all('tr')

        print(f"[INFO] : - len Table = {len(tbody[1:])}")

        for i, tr in enumerate(tbody[1:]):

            td = tr.find_all('td')

            sqldict['Name'].append(td[1].text.split('|')[0])

            sqldict['Address_1'].append(td[1].text.split('|')[1])

            sqldict['Phone'].append(td[2].text)

            sqldict['InternalID_1'].append(td[0].text)

            sqldict['InternalID_1_type'].append('INSURER NO')

            sqldict['InternalID_2'].append(td[4].text)

            sqldict['InternalID_2_type'].append('INSURER REGISTERED NUMBER')



            sqldict['Cntry'].append('ZA')

            sqldict['RegulationType'].append('Supervised')

            sqldict['ListProcessDate'].append(processdate)

            sqldict['RegCtry'].append(reg.split(' ')[0]) 

            sqldict['RegCode'].append(reg.split(' ')[1])

            sqldict['ListCode'].append(reg.split(' ')[-1])

        sqldict = bourange_same_length_array(sqldict)   



    elif reg == 'ZA FSCA 2' or reg == 'ZA FSCA 3' :

        opt = 'L' if reg == 'ZA FSCA 2' else 'F'

        Num = 7 if reg == 'ZA FSCA 2' else 6

        col_name = 2 if reg == 'ZA FSCA 2' else 1





        driver.find_element(By.XPATH, f'//*/select[@name="Local_Foreign"]/option[@value="{opt}"]').click()

        sleep(0.2)

        driver.find_element(By.XPATH, '//*/input[@value = "Submit"]').click() # Submit

        sleep(1)

        soup=BeautifulSoup(driver.page_source, 'html.parser')

        tbody = soup.find_all('tbody')[-1].find_all('tr')



        print(f"[INFO] : - Table containe = {len(tbody[1:])} rows")

        for i, tr in enumerate(tbody[1:]):

            td = tr.find_all('td')

            driver.find_element(By.XPATH, f'/html/body/form/center/table[2]/tbody/tr[{i+2}]/td[{Num}]/input').click() # click detail

            sleep(1)

            soup2=BeautifulSoup(driver.page_source, 'html.parser')

            for br in soup.find_all("br"):

                br.replace_with(" ")

            tables = soup2.find_all('table')

            Company_No = tables[0].find_all('tr')[-2].find('td').text if tables[0].find_all('tr')[-2].find('th').text == 'Company No' else ''

            Address = tables[1].find_all('tr')[0].find('td').text

            Phone = tables[1].find_all('tr')[1].find('td').text

            # Name = tables[2].find_all('tr')[-1].find_all('td')[1].text

            Status = tables[2].find_all('tr')[-1].find_all('td')[-2].text

            

            sqldict['Name'].append(td[col_name].text)

            sqldict['Address_1'].append(Address)

            sqldict['Phone'].append(Phone) if len(Phone) > 3 else sqldict['Phone'].append('')

            sqldict['InternalID_1'].append(Company_No)

            sqldict['InternalID_1_type'].append('Company Number') if len(Company_No)>0 else sqldict['InternalID_1_type'].append('')

            sqldict['InternalID_2'].append(td[0].text)

            sqldict['InternalID_2_type'].append('Manager No')

            sqldict['Cntry'].append('ZA')

            sqldict['RegulationType'].append(Status)

            sqldict['ListProcessDate'].append(processdate)

            sqldict['RegCtry'].append(reg.split(' ')[0]) 

            sqldict['RegCode'].append(reg.split(' ')[1])

            sqldict['ListCode'].append(reg.split(' ')[-1])

            sqldict = bourange_same_length_array(sqldict)   



            driver.back() # retour page precedante

            sleep(0.3)



            # if i ==12 :

            #     break

    elif reg == 'ZA FSCA 4':

        soup=BeautifulSoup(driver.page_source, 'html.parser')

        tbody = soup.find('tbody').find_all('tr')



        print(f"[INFO] : - len Table = {len(tbody[1:])}")

        for i, tr in enumerate(tbody[1:]):

            td = tr.find_all('td')



            if td[2].text.find('Registered') != -1 :



                sqldict['Name'].append(td[0].text)



                sqldict['Cntry'].append(td[1].text)

                sqldict['RegulationType'].append(td[2].text)

                sqldict['ListProcessDate'].append(processdate)

                sqldict['RegCtry'].append(reg.split(' ')[0]) 

                sqldict['RegCode'].append(reg.split(' ')[1])

                sqldict['ListCode'].append(reg.split(' ')[-1])

        sqldict = bourange_same_length_array(sqldict) 



    elif reg == 'ZA FSCA 5':



        driver.find_element(By.XPATH, '//*[@id="ctl00_PlaceHolderMain_ctl03__ControlWrapper_RichHtmlField"]').click()

        zip_file = check_dowload_files(tempfolder, "csv" )

        zip_filePath = os.path.join(tempfolder, zip_file)

        unzip(zip_filePath, tempfolder)

        sleep(1)

        xls_file = [f for f in os.listdir(tempfolder) if f.endswith('.xls') and os.path.isfile(os.path.join(tempfolder, f))][0]

        filePath = os.path.join(tempfolder, xls_file)



        with open(filePath, 'r', encoding='utf-8') as fichier: contenu = fichier.read()  # le fichier execl est une page HTML. Ne pas le lire avec pandas, sinon il y a un probleme de formatage 

        soup=BeautifulSoup(contenu, 'html.parser')

        table = soup.find('table').find_all('tr')



        print(f"[INFO] : - DataFrame containe = {len(table[3:])} rows")

        for i, tr in enumerate(table[3:]):

            td = tr.find_all('td')

            if len(td[1].text.strip())!=0 :

                sqldict['Name'].append(td[1].text.strip())

                sqldict['InternalID_1'].append(td[0].text.strip())

                sqldict['InternalID_1_type'].append('FUND NO')

                sqldict['RegulationType'].append('Registered')

                sqldict['ListProcessDate'].append(processdate)

                sqldict['RegCtry'].append(reg.split(' ')[0])

                sqldict['RegCode'].append(reg.split(' ')[1])

                sqldict['ListCode'].append(reg.split(' ')[-1])

            

        sqldict = bourange_same_length_array(sqldict) 

        

        for rem in os.listdir(tempfolder):

            os.remove(os.path.join(tempfolder, rem))



# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

#Moving the file to the output folder (this way it will be displayed in the Control Room)

sleep(3)


    